In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.datasets import mnist
from tensorflow.keras.regularizers import l1, l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Load MNIST
(x,y),(xt,yt)=mnist.load_data()
x=x.reshape(-1,784).astype("float32")/255
xt=xt.reshape(-1,784).astype("float32")/255

# Model
def M(reg=None,drop=0,lr=.001):
    m=Sequential([
        Dense(64,activation="relu",kernel_regularizer=reg,input_shape=(784,)),
        Dropout(drop),
        Dense(32,activation="relu"),
        Dense(10,activation="softmax")
    ])
    m.compile(optimizer=Adam(lr),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    return m

# Models
models={
    "L1":M(l1(.001)),
    "L2":M(l2(.001)),
    "Dropout":M(drop=.3),
    "EarlyStopping":M(),
    "Hyperparameter":M(drop=.2,lr=.0005)
}

results={}

# Training
for name,m in models.items():
    print("\nTraining:",name)

    if name=="EarlyStopping":
        es=EarlyStopping(monitor="val_loss",patience=2,
                         restore_best_weights=True)
        h=m.fit(x,y,epochs=10,validation_split=.2,
                callbacks=[es],verbose=0)
    else:
        h=m.fit(x,y,epochs=10,validation_split=.2,verbose=0)

    acc=m.evaluate(xt,yt,verbose=0)[1]
    results[name]=(h,acc)
    print("Test Accuracy:",round(acc,4))

# Accuracy graph
for name,(h,acc) in results.items():
    plt.plot(h.history["accuracy"],label=name)

plt.title("Regularization - Accuracy Comparison")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

# Loss graph
for name,(h,acc) in results.items():
    plt.plot(h.history["loss"],label=name)

plt.title("Regularization - Loss Comparison")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

# Final comparison
print("\nPerformance Comparison")
for name,(h,acc) in results.items():
    print(name,"-> Accuracy:",round(acc,4),
          "Loss:",round(h.history["loss"][-1],4))